### Enterprise Energy Analytics: Automated Multi-Format ETL Data Pipeline

#### Executive Summary & Business Context
In the utility and energy sector, tracking changes in power generation capability, sector consumption, and regional pricing dynamics is essential for market forecasting, procurement, and regulatory compliance. Previously, internal analytics teams relied on manual quarterly data retrieval and ad-hoc cleaning workflows—a labor-intensive process requiring days of effort that introduced human error and created severe latency in reporting.

To eliminate this operational bottleneck, this project delivers an automated, production-grade Extract, Transform, Load (ETL) data pipeline. Built in Python using `pandas`, the pipeline ingests heterogeneous data sources (relational tabular records and deeply nested JSON structures), executes domain-specific cleaning and temporal parsing, and exports standardized datasets across columnar (`.parquet`) and delimited (`.csv`) storage sinks.

#### Data Dictionary

| Field | Type | Description |
| :--- | :--- | :--- |
| `period` | `str` | Combined observation timestamp encoding month and year |
| `stateid` | `str` | Two-letter state postal abbreviation |
| `stateDescription` | `str` | Full geographic state name |
| `sectorid` | `str` | Energy consumer sector identifier |
| `sectorName` | `str` | Consumer category classification (`residential`, `transportation`, etc.) |
| `price` | `float` | Retail electricity rate |
| `price-units` | `str` | Measurement unit denomination (e.g., cents per kilowatt-hour) |

In [ ]:
# Import relevant libraries
import pandas as pd
import json

#### Data Extraction
Implementing specialized extraction modules:
1. `extract_tabular_data()`: Handles multi-format tabular ingestion with validation for `.csv` and `.parquet` file types.
2. `extract_json_data()`: Ingests raw JSON documents and applies schema flattening via `pd.json_normalize()` to convert nested hierarchies into a flat tabular structure.

In [ ]:
def extract_tabular_data(file_path: str):
    """Extract data from a tabular file_format, with pandas."""
    if file_path.endswith('.csv'):
        return pd.read_csv(file_path)
    elif file_path.endswith('.parquet'):
        return pd.read_parquet(file_path)
    else:
        raise Exception("Warning: Invalid file extension. Please try with .csv or .parquet!")

def extract_json_data(file_path):
    """Extract and flatten data from a JSON file."""
    with open(file_path, 'r') as file:
        data = json.load(file)
    return pd.json_normalize(data)

#### Data Transformation
Executing core business logic on raw electricity sales transactions:
* **Missing Value Removal:** Dropping invalid records containing null values in the `price` column.
* **Cohort Isolation:** Filtering records to isolate high-priority consumer classes: `residential` and `transportation`.
* **Date Parsing:** Decomposing the composite `period` string into discrete `month` (leading 4 characters) and `year` (trailing 2 characters) components.
* **Dimensionality Selection:** Projecting the final cleaned schema across `['year', 'month', 'stateid', 'price', 'price-units']`.

In [ ]:
def transform_electricity_sales_data(raw_data: pd.DataFrame):
    # Drop records with NA in the 'price' column
    raw_data.dropna(subset=['price'], inplace=True)

    # Only keep records with a sector name of residential and transportation
    raw_data = raw_data[raw_data['sectorName'].isin(['residential', 'transportation'])]

    # Create month column using the first 4 letters of the values in period
    raw_data['month'] = raw_data['period'].str[:4]

    # Create year column using the last 2 characters of the values in period
    raw_data['year'] = raw_data['period'].str[-2:]

    # Filter for specified columns and return the final DataFrame
    final_columns = ['year', 'month', 'stateid', 'price', 'price-units']
    return raw_data[final_columns]


#### Data Storage
Persisting sanitized DataFrames into either delimited (`.csv`) or columnar (`.parquet`) storage with index exclusion and runtime extension validation.

In [ ]:
def load(dataframe: pd.DataFrame, file_path: str):
    """Load a DataFrame to a file in either CSV or Parquet format."""
    if file_path.endswith(".csv"):
        dataframe.to_csv(file_path, index=False)
    elif file_path.endswith(".parquet"):
        dataframe.to_parquet(file_path, index=False)
    else:
        raise Exception(f"Warning: {file_path} is not a valid file type. Please try again!")

#### End-to-End Pipeline Execution & Integration Testing
Orchestrating the complete extract, transform, and load routine across both the capability and sales data pipelines:
1. Ingesting and flattening nested capability records.
2. Ingesting raw retail sales transactions.
3. Transforming sales records against business rules.
4. Exporting serialized artifacts to `.parquet` and `.csv` persistent targets.

In [ ]:
raw_electricity_capability_df = extract_json_data("electricity_capability_nested.json")
raw_electricity_sales_df = extract_tabular_data("electricity_sales.csv")

cleaned_electricity_sales_df = transform_electricity_sales_data(raw_electricity_sales_df)

load(raw_electricity_capability_df, "loaded__electricity_capability.parquet")
load(cleaned_electricity_sales_df, "loaded__electricity_sales.csv")